# Executor de experimentos

- Usado para a execução de cada experimento no ambiente do Google Colab.
- Cada experimento foi realizado em três rodadas com seeds diferentes
- Os resultados de cada experimento foram salvas em arquivos no Google Drive e num banco de dados

Verificação do ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Thu May  7 04:35:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

Clonagem do repositório atualizado no GitHub

In [2]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

/content
fatal: destination path 'classification-of-medical-images-using-cnn' already exists and is not an empty directory.
/content/classification-of-medical-images-using-cnn


Montagem do Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Configuração dos parâmetros do experimento

In [4]:
EXPERIMENT_NAME = "resnet-no-data-aug"
BASE_MODEL = "resnet"
NORMALIZATION = "preprocess-input"
DATA_AUG = False
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = False
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

Criação da engine do banco de dados
- Foi usado um banco de dados sqlite hospedado também no Google Drive

In [5]:
from src.db import insert_run, get_engine

DB_PATH = f"sqlite:///{BASE_PATH}/experiments.db"

engine = get_engine(DB_PATH)

Realização das três rodadas do experimento com:
- Treinamento do modelo
- Teste do modelo treinado
- Persitência dos parâmetros e dos resulados da rodada em banco de dados

In [6]:
from src.train import train_pipeline
from src.test import test_pipeline

for i in range(3):
    run_id = i + 1
    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
    )

    metrics = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run-{run_id}",
        config=config,
        metrics=metrics,
    )

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 75s 361ms/step - AUC: 0.9814 - accuracy: 0.9365 - loss: 0.1517 - val_AUC: 1.0000 - val_accuracy: 0.5625 - val_loss: 0.8627
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 14s 88ms/step - AUC: 0.9955 - accuracy: 0.9722 - loss: 0.0753 - val_AUC: 1.0000 - val_accuracy: 0.8750 - val_loss: 0.3214
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 14s 86ms/step - AUC: 0.9965 - accuracy: 0.9749 - loss: 0.0655 - val_AUC: 1.0000 - val_accuracy: 1.0000 - val_loss: 0.0765
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 14s 84ms/step - AUC: 0.9983 - accuracy: 0.9837 - loss: 0.0438 - val_AUC: 1.0000 - val_accuracy: 0.8125 - val_loss: 0.2555
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 14s 83ms/step - AUC: 0.9990 - accuracy: 0.9893 - loss: 0.0332 - val_AUC: 1.0000 - val_accuracy: 0.8125 - val_loss: 0.2950
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 14s 83ms/step - AUC: 0.9993 - accuracy: 0.9900 - loss: 0.0284 - val_A

Instala DagsHub para versionamento dos resultados dos experimentos
- O DagsHub é uma plataforma de versionamento de dados e modelos, similar ao GitHub, mas focada em projetos de ciência de dados e machine learning
- Loga no DagsHub com Secrets do Google Colab
- Faz upload da pasta com os resultados do último experimento feito

In [7]:
%pip install -q dvc dagshub

import dagshub
from google.colab import userdata

dagshub.auth.add_app_token(token=userdata.get("DAGSHUB_TOKEN"))

RESULTS_RELATIVE_PATH = "results/" + EXPERIMENT_NAME
LOCAL_PATH = BASE_PATH + "/" + RESULTS_RELATIVE_PATH

dagshub.upload_files(
    "amartinsmg/classification-of-medical-images-using-cnn",
    local_path=LOCAL_PATH,
    remote_path=RESULTS_RELATIVE_PATH,
)

Accessing as amartinsmg

Output()

Directory upload complete, uploaded 9 files to 
https://dagshub.com/amartinsmg/classification-of-medical-images-using-cnn/src/main/results%2Fresnet-no-data-aug